# Mamba Design

Designing Mamba architecture based on this [block diagram](https://www.ibm.com/think/topics/mamba-model). 


In [9]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim

Before, we had to create our own custom tokenizer, based on each char in our vocab_size. We will no longer do this! Let's use a pre-trained tokenizer from **HuggingFace** with all the fun bells and whistles.

In [10]:
from transformers import AutoTokenizer # HuggingFace 

# mamba uses "EleutherAI/gpt-neox-20b" tokenizer
tokenizer_name = "EleutherAI/gpt-neox-20b"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
tokenizer.model_max_length = 2048 # max seq length of Mamba (example)
tokenizer.pad_token = tokenizer.eos_token # explicitly say when things end
test_input = "Hello world!"
tensor_input = tokenizer(test_input, return_tensors="pt")
print(tensor_input["input_ids"].shape) # 1 x L
print(tokenizer.decode(tensor_input["input_ids"][0]))
print(f"Vocab size: {tokenizer.vocab_size}")

torch.Size([1, 3])
Hello world!
Vocab size: 50254


We will also test out the train-test split and get_batch standards in **HuggingFace**. Should be of good use to us long down the road when we want to quickly set these features up for our model of choice.

Make sure to have a good understanding of:
- Iterables vs Iterators (Native Python)
- Dataset vs DatasetDict (HF)
- Tensors (PT)
- Native collectibles (list, dict, set, etc.) 

In [11]:
from datasets import Dataset, load_dataset

# hardcoded set of strings (data)
# raw_data = {
#     "text": [
#         "PyTorch makes tensor math easy.",
#         "Hugging Face standardizes the data pipelines.",
#         "NanoGPT is a great learning resource.",
#         "Attention is all you need."
#     ]
# }
# hf_dataset = Dataset.from_dict(raw_data)
# print(hf_dataset)

# now, we will use load_dataset to get whatever large chunk of data we want for pretraining!
# if we don't shuffle, we're going to go through the same row group A -> row group B -> ...
raw_data_stream = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", streaming=True)
# shuffle from the large dataset for a little bit of randomness :)
# buffer_size says "hey, i want to pre-install 10,000 rows of data and put them into DRAM"
shuffled_stream = raw_data_stream.shuffle(buffer_size=10000)
print(shuffled_stream)

IterableDatasetDict({
    train: IterableDataset({
        features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
        num_shards: 14
    })
})


In [12]:
# take(100) says "hey, out of the entire buffer, i want to stop at just 100 rows if i iterate on it"
# for pretraining, we will rely on the ENTIRE pool of 10k. for now, let's do a small test w/ 100
raw_data = list(shuffled_stream['train'].take(100))
print(raw_data)
print(len(raw_data))

[{'text': 'The War Museum (Militärhistorisches Kriegsmuseum) in Dresden offers a very rare glimpse in Germany into warfare, military history, and fighting equipment. The museum shows how the military, armies, and war influenced politics and society, and vice versa. Some of the most famous large items in the museum include a V2 flying bomb, Germany’s first submarine, the Horch cabriolet used by Choltitz and later Charles de Gaulle, and the landing craft that returned the first German in space to earth.\nThe German Military History Museum in Dresden\nThe Militärhistorisches Museum der Bundeswehr – Military History Museum of the Federal (German) Armed Forces in Dresden, Saxony, is often referred to as the Kriegsmuseum or War Museum. This museum is one of very few in Germany that actually has German war equipment from the First and Second World Wars on display.\nThe museum follows the current standard German approach to controversial historical buildings and issues: get an international st

In [13]:
# convert this list of dictionaries into a dataset
hf_dataset = Dataset.from_list(raw_data)
print(hf_dataset)

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
    num_rows: 100
})


In [14]:
# train-test split
hf_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(hf_split)
print(hf_split['train']['text'][0]) # one row from our text

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
        num_rows: 90
    })
    test: Dataset({
        features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
        num_rows: 10
    })
})
What is a SCADA sytem?
Supervisory control and data acquisition (SCADA) is a computer system used to monitor and control plant processes.
The biggest manufacturing companies in the world are also known to be the most data-driven establishments. In an age of growing technological capabilities, the importance of collecting data is pushed to the limit with the use of systems such as SCADA.
By collecting and monitoring real-time data, SCADA shows an overview of how each key equipment in the plant is performing. Sensors on the equipment send signals through remote terminal units (RTU) and programmable logic controllers (PLC).

In [19]:
# getting a batch with collator (this time for causal LLM)
from transformers import DataCollatorForLanguageModeling # define get_batch
from torch.utils.data import DataLoader # use get_batch 

def tokenize(dataset : Dataset):
    encode = lambda dataset : tokenizer(dataset['text'], truncation=True)
    return dataset.map(encode, batched=True) # batched -> parallel processing

# HF uses the map function to add tokenized input to dataset (with map)
res = tokenize(hf_split['train']) 
print(res)
print(res['text'][0])
print(res['input_ids'][0])
print(len(res['input_ids'][0])) # great, now we did some pre-processing of our input data!

Map: 100%|██████████| 90/90 [00:00<00:00, 1544.41 examples/s]

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score', 'input_ids', 'attention_mask'],
    num_rows: 90
})
What is a SCADA sytem?
Supervisory control and data acquisition (SCADA) is a computer system used to monitor and control plant processes.
The biggest manufacturing companies in the world are also known to be the most data-driven establishments. In an age of growing technological capabilities, the importance of collecting data is pushed to the limit with the use of systems such as SCADA.
By collecting and monitoring real-time data, SCADA shows an overview of how each key equipment in the plant is performing. Sensors on the equipment send signals through remote terminal units (RTU) and programmable logic controllers (PLC). This gives the SCADA system the ability to pinpoint anomalies in system functions based on the collected data, thereby allowing the user to promptly take action on the issue.
SCADA allo

In [33]:
# now, we need to "clean up" the data so it's compatible with mamba
# make sure when we convert to tensor it adheres to the tokenizer rules we established earlier
# this includes max length, proper padding in tensors, sizing, end-of-sequence char, etc.
hf_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)
# small note: padding to multiple of 8 --> takes max in batch and then pads to that multiple of 8
loader = DataLoader(res['input_ids'], collate_fn=hf_collator, batch_size=8, shuffle=True)
# this should result in a batch from our training/testing!
batch_example = next(iter(loader))['input_ids'] # (B, L)
print(batch_example[0])
print(tokenizer.decode(batch_example[0]))
print(batch_example.shape)

tensor([23262, 35826,  2213,  ...,     0,     0,     0])
Ray Hubbard Reservoir - 2012 Survey Report
Prepared by Raphael Brock and Thomas Hungerford
Inland Fisheries Division
District 2-D, Fort Worth, Texas
This is the authors' summary from a 32-page report. For a copy of the complete report, use the download link in the sidebar.
Fish populations in Ray Hubbard Reservoir were surveyed in 2009, 2010, 2011, and 2012 using electrofishing, 2011 and 2013 using gill netting, and 2009 and 2012 using trap netting. This report summarizes the results of the surveys and contains a management plan for the reservoir based on those findings.
Ray Hubbard Reservoir is a 22,745-acre impoundment constructed on the East Fork of the Trinity River by the City of Dallas in 1968 to provide water for municipal, industrial, and recreational purposes. Ray Hubbard Reservoir lies within Dallas, Collin, Rockwall and Kaufman counties. The reservoir is part of the Dallas-Ft. Worth metroplex. The reservoir has a 1,074

### Creating our model

After figuring out how to get standard datasets, pre-processing, and getting our batches, we can finally move on to creating our first generic Mamba model :)

Ingredients:
- Linear Projection (2*D)
- 1D convolution (mixing)
- Residual + Skip Connection (Residual-after-add)
- Non-linearity (SiLU)
- SSM block (discretization, recurrence, etc.)
- FFWD (xt --> B_t, C_t, delta_t)

In [46]:
# TESTING OUT SSM EQUATIONS!

# Configuration
B = 8
D = 768
L = 128  
N = 24

# 1. Initialize RAW inputs (using floats!)
X = torch.randn(B, L, D)
A = torch.randn(D, N)
B_tensor = torch.randn(B, L, N) # Renamed so it doesn't overwrite B=8
C = torch.randn(B, L, N)
# delta is a time step, so we use abs() to keep it strictly positive
delta = torch.abs(torch.randn(B, L, D)) 

# ==========================================
# PHASE 1: PARALLEL DISCRETIZATION
# ==========================================
delta_A = torch.einsum('BLD,DN->BLDN', delta, A)
A_bar = torch.exp(delta_A)

delta_B = torch.einsum('BLD,BLN->BLDN', delta, B_tensor)
zoh_factor = (A_bar - 1.0) / delta_A 
B_bar = zoh_factor * delta_B

# ==========================================
# PHASE 2: SEQUENTIAL SCAN
# ==========================================
Y = []
H_prev = None

if H_prev is not None:
    H_current = H_prev
else:
    H_current = torch.zeros((B, D, N))
    
for t in range(L):
    # Slice out the current time step
    X_t = X[:, t, :]          # (B, D)
    C_t = C[:, t, :]          # (B, N)
    A_bar_t = A_bar[:, t, :, :]  # (B, D, N)
    B_bar_t = B_bar[:, t, :, :]  # (B, D, N)
    
    # State update
    U_decay = torch.einsum('BD,BDN->BDN', X_t, B_bar_t)
    H_decay = torch.einsum('BDN,BDN->BDN', A_bar_t, H_current)
    H_current = U_decay + H_decay
    
    # Output projection
    Y_next = torch.einsum('BN,BDN->BD', C_t, H_current)
    Y.append(Y_next)
    
Y_stacked = torch.stack(Y, dim=1)

# Print shapes to prove the math worked flawlessly
print(f"Final State Shape:  {H_current.shape}  | Expected: [{B}, {D}, {N}]")
print(f"Output Seq Shape:   {Y_stacked.shape}  | Expected: [{B}, {L}, {D}]")

Final State Shape:  torch.Size([8, 768, 24])  | Expected: [8, 768, 24]
Output Seq Shape:   torch.Size([8, 128, 768])  | Expected: [8, 128, 768]


(1) SSM Equation

$h'(t) = \mathbf{A}h(t) + \mathbf{B}x(t) \\
y(t) = \mathbf{C}h(t)$


(2) ZOH Discretization

$\bar{\mathbf{A}} = \exp(\Delta \mathbf{A}) \\
\bar{\mathbf{B}} = (\Delta \mathbf{A})^{-1} (\exp(\Delta \mathbf{A}) - \mathbf{I}) \cdot \Delta \mathbf{B}$

(3) Discrete SSM

$h_t = \bar{\mathbf{A}}h_{t-1} + \bar{\mathbf{B}}x_t \\
y_t = \mathbf{C}h_t$

Note that A is a diagonal matrix. Therefore, inverse(x) = 1/x.

In [47]:
# == input ==
# X (B, L, D)
# A (D, N)
# B (B, L, N)
# C (B, L, N)
# delta (B, L, D)
# H_prev (B, D, N) <- Notice this is just the single state from before the sequence started
# == output ==
# H_next (B, D, N) <- The final state to pass to the next chunk
# Y (B, L, D)
class SSMBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config 
    
    def forward(self, X, A, B, C, delta, H_prev=None):
        # ==========================================
        # 1. PARALLEL DISCRETIZATION
        # ==========================================
        delta_A = torch.einsum('BLD,DN->BLDN', delta, A)
        A_bar = torch.exp(delta_A)
        
        delta_B = torch.einsum('BLD,BLN->BLDN', delta, B)
        zoh_factor = (A_bar - 1.0) / delta_A 
        B_bar = zoh_factor * delta_B
        
        # ==========================================
        # 2. SEQUENTIAL SCAN
        # ==========================================
        Y = []
        
        # Initialize the rolling state
        if H_prev is not None:
            H_current = H_prev
        else:
            H_current = torch.zeros((self.config.B, self.config.D, self.config.N), device=X.device)
            
        # Step through every token in the sequence (t)
        for t in range(self.config.L):
            # Slice out the inputs and parameters for ONLY this exact moment in time
            X_t = X[:, t, :]          # (B, D)
            C_t = C[:, t, :]          # (B, N)
            A_bar_t = A_bar[:, t, :, :]  # (B, D, N)
            B_bar_t = B_bar[:, t, :, :]  # (B, D, N)
            
            # SSM equations (Notice the strings no longer have 'L')
            U_decay = torch.einsum('BD,BDN->BDN', X_t, B_bar_t)
            H_decay = torch.einsum('BDN,BDN->BDN', A_bar_t, H_current)
            
            # Update the rolling state
            H_current = U_decay + H_decay
            
            # Calculate output and append to our list
            Y_next = torch.einsum('BN,BDN->BD', C_t, H_current)
            Y.append(Y_next)
            
        # Stack the list of (B, D) tensors back into a (B, L, D) tensor
        Y_stacked = torch.stack(Y, dim=1)
        
        # Return the final state and the full sequence output
        return H_current, Y_stacked

(4) Selective Mechanism

$\mathbf{B}_t = \text{Linear}_B(x_t) \\
\mathbf{C}_t = \text{Linear}_C(x_t) \\
\Delta_t = \text{softplus}(\text{Linear}_\Delta(x_t)) \\
\bar{\mathbf{A}}_t, \bar{\mathbf{B}}_t = \text{discretize}(\Delta_t, \mathbf{A}, \mathbf{B}_t)$

In [ ]:
# Up Projection -> SSM + Gating -> Down Projection -> Residual + Skip
class MixerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # for main path
        self.up_projection_1 = nn.Linear(self.config.D_outer, self.config.D, bias=False)
        self.silu_1 = nn.SiLU(self.config.D)
        self.ll = nn.Linear(self.config.D, self.config.D*3) # out: delta, B, C
        self.sp = nn.Softplus() # for delta
        self.ssm = SSMBlock(config=self.config)
        # for gated path
        self.up_projection_2 = nn.Linear(self.config.D_outer, self.config.D, bias=False)
        self.silu_2 = nn.SiLU(self.config.D)
        # after gated * main 
        self.down_projection = nn.Linear(self.config.D, self.config.D_outer)
        self.layernorm = nn.LayerNorm()
    
    def forward(self, X):
        X_proj = self.up_projection_1(X)
        # idk how to do conv 1d here
        X_conv = X_proj
        X_final = self.silu_1(X_conv)
        delta_i, B, C = self.ll(X_final)
        delta = self.sp(delta_i)
        Y_ssm = self.ssm(delta, B, C)
        Y_proj = self.down_projection(Y_ssm)
        Y = self.layernorm(X + Y_proj)
        return Y

In [ ]:
class MambaModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config=config # should be a dict of our arguments
        self.layers = nn.ModuleList([MixerBlock(config=self.config) for i in self.config.n_layers])
    
    def forward(self, X):
        return self.layers(X)